# South Africa Public Procument Analysis

In [2]:
# Data loading & Initial Inspection

# Import required libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Display settings - show all columns when printing the dataframe
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("All libraries imported successfully")
print(f"Analysis started:{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

All libraries imported successfully
Analysis started:2026-04-11 20:02:42


In [3]:
# Load all raw datasets

print("Loading datasets from 00_raw_data/ ...\n")
print("-"*80)

# Load each dataset
main = pd.read_csv("../00_raw_data/main.csv", low_memory=False)
awards = pd.read_csv("../00_raw_data/awards.csv", low_memory=False)
awards_suppliers = pd.read_csv("../00_raw_data/awards_suppliers.csv", low_memory=False)
contracts = pd.read_csv("../00_raw_data/contracts.csv", low_memory=False)
contracts_documents = pd.read_csv("../00_raw_data/contracts_documents.csv", low_memory=False)
parties = pd.read_csv("../00_raw_data/parties.csv", low_memory=False)
tender_documents = pd.read_csv("../00_raw_data/tender_documents.csv", low_memory=False)
tender_tenderers = pd.read_csv("../00_raw_data/tender_tenderers.csv", low_memory=False)

print("All datasets loaded successfully!\n")


Loading datasets from 00_raw_data/ ...

--------------------------------------------------------------------------------
All datasets loaded successfully!



In [4]:
def inspect_dataset(df, name):
    """
    Complete inspection of a single dataset.
    """
    
    print("\n" + "=" * 100)
    print(f"📁 DATASET: {name}")
    print("=" * 100)
    
    # 1. BASIC SHAPE
    rows = len(df)
    cols = len(df.columns)
    print(f"\n📏 SHAPE: {rows:,} rows × {cols} columns")
    
    # 2. MEMORY USAGE
    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"💾 MEMORY USAGE: {memory_mb:.2f} MB")
    
    # 3. COLUMN DETAILS
    print(f"\n📋 COLUMNS ({cols} total):")
    print("-" * 100)
    
    # Simple header without complex formatting
    header = "  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %"
    print(header)
    print("-" * 100)
    
    for i, col in enumerate(df.columns, 1):
        col_str = str(col)
        dtype = str(df[col].dtype)
        non_null = df[col].count()
        null_count = df[col].isnull().sum()
        
        if rows > 0:
            null_pct = (null_count / rows) * 100
        else:
            null_pct = 0
        
        # Truncate column name if too long
        if len(col_str) > 40:
            col_display = col_str[:37] + "..."
        else:
            col_display = col_str
        
        # Build the line without f-string formatting that causes errors
        line = f"{i:3d} | {col_display:<40} | {dtype:<13} | {non_null:11,} | {null_count:11,} | {null_pct:5.1f}%"
        print(line)
    
    # 4. DUPLICATE ROWS
    print(f"\n📋 DUPLICATE ROWS:")
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        dup_pct = (dup_count / rows) * 100 if rows > 0 else 0
        print(f"   ⚠️ {dup_count:,} duplicate rows ({dup_pct:.2f}% of data)")
    else:
        print(f"   ✅ No duplicate rows")
    
    # 5. COMPLETELY EMPTY COLUMNS
    all_null_cols = df.columns[df.isnull().all()].tolist()
    if all_null_cols:
        print(f"\n⚠️ COMPLETELY EMPTY COLUMNS (100% null):")
        for col in all_null_cols:
            print(f"   - {str(col)}")
    else:
        print(f"\n✅ No completely empty columns")
    
    # 6. POTENTIAL IDENTIFIER COLUMNS
    id_keywords = ['id', 'code', 'number', 'reference', 'ref', 'key']
    id_cols = []
    
    for col in df.columns:
        col_str = str(col).lower()
        if any(keyword in col_str for keyword in id_keywords):
            unique_vals = df[col].nunique()
            if rows > 0:
                uniqueness = (unique_vals / rows) * 100
            else:
                uniqueness = 0
            id_cols.append((str(col), unique_vals, uniqueness))
    
    if id_cols:
        print(f"\n🔑 POTENTIAL IDENTIFIER COLUMNS:")
        for col, unique, pct in id_cols[:10]:
            print(f"   - {col:<40} | {unique:,} unique values | {pct:.1f}% unique")
        if len(id_cols) > 10:
            print(f"   ... and {len(id_cols) - 10} more")
    
    
    # 8. DATA TYPES SUMMARY
    print(f"\n📌 DATA TYPES SUMMARY:")
    dtype_counts = df.dtypes.astype(str).value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   {dtype}: {count} columns")
    
    print("\n" + "=" * 100)
    print(f"✅ INSPECTION COMPLETE FOR: {name}")
    print("=" * 100)
    
    # Return summary
    return {
        'name': name,
        'rows': rows,
        'columns': cols,
        'memory_mb': round(memory_mb, 2),
        'duplicates': dup_count,
        'missing_cells': int(df.isnull().sum().sum()),
        'id_cols': len(id_cols)
    }



**Cell 1: Inspect Main Dataset**

In [21]:
main_summary = inspect_dataset(main, "1. Main")


📁 DATASET: 1. Main

📏 SHAPE: 43,384 rows × 36 columns
💾 MEMORY USAGE: 97.62 MB

📋 COLUMNS (36 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | object        |      43,384 |           0 |   0.0%
  2 | tag                                      | object        |      43,384 |           0 |   0.0%
  3 | date                                     | object        |      43,384 |           0 |   0.0%
  4 | ocid                                     | object        |      43,384 |           0 |   0.0%
  5 | initiationType                           | object        |      43,384 |           0 |   0.0%
  6 | language                                 | object        |      43,384 |           0 |

**Cell 2: Awards**

In [22]:
awards_summary = inspect_dataset(awards, "2. AWARDS")


📁 DATASET: 2. AWARDS

📏 SHAPE: 8,190 rows × 8 columns
💾 MEMORY USAGE: 3.50 MB

📋 COLUMNS (8 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | int64         |       8,190 |           0 |   0.0%
  2 | title                                    | object        |       8,190 |           0 |   0.0%
  3 | description                              | object        |       8,181 |           9 |   0.1%
  4 | status                                   | object        |       8,190 |           0 |   0.0%
  5 | value_amount                             | float64       |       8,190 |           0 |   0.0%
  6 | value_currency                           | object        |       8,190 |           0 |  

**Cell 3: Awards_suppliers**

In [23]:
awards_suppliers_summary = inspect_dataset(awards_suppliers, "3. AWARDS_SUPPLIERS")


📁 DATASET: 3. AWARDS_SUPPLIERS

📏 SHAPE: 8,190 rows × 5 columns
💾 MEMORY USAGE: 2.07 MB

📋 COLUMNS (5 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | int64         |       8,190 |           0 |   0.0%
  2 | name                                     | object        |       8,190 |           0 |   0.0%
  3 | main_ocid                                | object        |       8,190 |           0 |   0.0%
  4 | main_id                                  | object        |       8,190 |           0 |   0.0%
  5 | awards_id                                | int64         |       8,190 |           0 |   0.0%

📋 DUPLICATE ROWS:
   ✅ No duplicate rows

✅ No completely empty columns

🔑 POTENTIA

**Cell 4: Contracts**

contracts_summary = inspect_dataset(contracts, "4. CONTRACTS")

**Cell 5: Contracts_suppliers**

In [5]:
contracts_suppliers_summary = inspect_dataset(contracts_documents, "5. CONTRACTS_SUPPLIERS")


📁 DATASET: 5. CONTRACTS_SUPPLIERS

📏 SHAPE: 1,230 rows × 11 columns
💾 MEMORY USAGE: 1.03 MB

📋 COLUMNS (11 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | object        |       1,230 |           0 |   0.0%
  2 | title                                    | object        |       1,230 |           0 |   0.0%
  3 | format                                   | object        |       1,230 |           0 |   0.0%
  4 | language                                 | object        |       1,230 |           0 |   0.0%
  5 | description                              | object        |       1,230 |           0 |   0.0%
  6 | dateModified                             | object        |       1,230 | 

**Cell 6: Parties**

In [27]:
parties_summary = inspect_dataset(parties, "6. PARTIES")


📁 DATASET: 6. PARTIES

📏 SHAPE: 8,190 rows × 12 columns
💾 MEMORY USAGE: 6.07 MB

📋 COLUMNS (12 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | int64         |       8,190 |           0 |   0.0%
  2 | name                                     | object        |       8,190 |           0 |   0.0%
  3 | roles                                    | object        |       8,190 |           0 |   0.0%
  4 | address_countryName                      | object        |       8,190 |           0 |   0.0%
  5 | identifier_legalName                     | object        |       8,190 |           0 |   0.0%
  6 | contactPoint_url                         | object        |       8,190 |           0 

**Cell 7: Tender_documents**

In [28]:
tender_documents_summary = inspect_dataset(tender_documents, "7. TENDER_DOCUMENTS")


📁 DATASET: 7. TENDER_DOCUMENTS

📏 SHAPE: 44,313 rows × 12 columns
💾 MEMORY USAGE: 44.00 MB

📋 COLUMNS (12 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | object        |      44,313 |           0 |   0.0%
  2 | url                                      | object        |      44,313 |           0 |   0.0%
  3 | title                                    | object        |      44,313 |           0 |   0.0%
  4 | format                                   | object        |      44,313 |           0 |   0.0%
  5 | language                                 | object        |      44,313 |           0 |   0.0%
  6 | description                              | object        |      44,313 |  

**Cell 8: Tender_tenderers**

In [29]:
tender_tenderers_summary = inspect_dataset(tender_tenderers, "8. TENDER_TENDERERS")


📁 DATASET: 8. TENDER_TENDERERS

📏 SHAPE: 2,179 rows × 5 columns
💾 MEMORY USAGE: 0.61 MB

📋 COLUMNS (5 total):
----------------------------------------------------------------------------------------------------
  # | Column Name                              | Data Type      | Non-Null     | Nulls        | Null %
----------------------------------------------------------------------------------------------------
  1 | id                                       | int64         |       2,179 |           0 |   0.0%
  2 | name                                     | object        |       2,175 |           4 |   0.2%
  3 | main_ocid                                | object        |       2,179 |           0 |   0.0%
  4 | main_id                                  | object        |       2,179 |           0 |   0.0%
  5 | tender_id                                | int64         |       2,179 |           0 |   0.0%

📋 DUPLICATE ROWS:
   ✅ No duplicate rows

✅ No completely empty columns

🔑 POTENTIA